In [ ]:
# Robust MLflow logging + optional TensorBoard artifact capture + UC registration
import os, glob
import numpy as np
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

import torch
import torch.nn as nn

# 0) Resolve run name & candidate TensorBoard/Artifact directories
try:
    run_name = run_config.name  # from TorchTrainer RunConfig
except Exception:
    run_name = "ray_tabular_run"

candidate_dirs = set()

# Try Ray result.path (if available)
try:
    ray_result_path = result.path  # e.g. file:///dbfs/tmp/...
    ray_result_path_fs = ray_result_path[len("file://"):] if ray_result_path.startswith("file://") else ray_result_path
    if os.path.isdir(ray_result_path_fs):
        candidate_dirs.add(ray_result_path_fs)
except Exception:
    pass

# Legacy Ray artifact areas on DB envs (best-effort)
pattern_driver = f"/local_disk0/tmp/ray-*/session_*/artifacts/*/{run_name}/driver_artifacts"
for p in glob.glob(pattern_driver):
    if os.path.isdir(p):
        candidate_dirs.add(p)

# Also look under storage_path from RunConfig
try:
    storage_root = run_config.storage_path
    if storage_root and os.path.isdir(storage_root):
        for p in glob.glob(os.path.join(storage_root, "**", run_name), recursive=True):
            if os.path.isdir(p):
                candidate_dirs.add(p)
except Exception:
    pass

def latest_mtime(path: str) -> float:
    """Return latest modification time in a directory tree."""
    mt = os.path.getmtime(path)
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try:
                mt = max(mt, os.path.getmtime(fp))
            except Exception:
                pass
    return mt

def has_tb_events(d: str) -> bool:
    """Check if directory contains TensorBoard event files."""
    if not d or not os.path.isdir(d):
        return False
    return bool(glob.glob(os.path.join(d, "**", "events.out.tfevents*"), recursive=True))

# Choose freshest candidate or fallback
experiment_log_dir = None
if candidate_dirs:
    experiment_log_dir = max(candidate_dirs, key=latest_mtime)

if experiment_log_dir is None:
    experiment_log_dir = "/tmp/ray_tabular_airbnb_tensorboard"
    os.makedirs(experiment_log_dir, exist_ok=True)
    print("No TensorBoard events found; using empty directory:", experiment_log_dir)
else:
    if has_tb_events(experiment_log_dir):
        print("TensorBoard events directory:", experiment_log_dir)
    else:
        print(
            "Artifact directory found but no TensorBoard event files; "
            "TensorBoard UI may appear empty:",
            experiment_log_dir,
        )

# 1) Build MLflow params/metrics from training config + Ray result
mlflow_params = {}
if "train_config" in globals() and isinstance(train_config, dict):
    # Drop large/irrelevant objects if present
    mlflow_params = {
        k: v for k, v in train_config.items()
        if k not in {"val_ds", "X_val", "y_val"}
    }

# Add scaling info if present
try:
    mlflow_params.update({
        "num_workers": scaling_config.num_workers,
        "use_gpu": scaling_config.use_gpu,
        "resources_per_worker": scaling_config.resources_per_worker,
    })
except Exception:
    pass

# Extract a few metrics from Ray result (regression case)
log_metrics = {}
try:
    if result and isinstance(result.metrics, dict):
        if result.metrics.get("best_metric") is not None:
            # In our training loop best_metric is based on val_loss (negative or raw)
            log_metrics["best_metric"] = float(result.metrics["best_metric"])
except Exception:
    pass

# 2) Log params/metrics + optional TensorBoard artifacts to MLflow
mlflow.end_run()
with mlflow.start_run(run_name=run_name):
    if mlflow_params:
        mlflow.log_params(mlflow_params)
    for k, v in log_metrics.items():
        mlflow.log_metric(k, v)

    if experiment_log_dir and has_tb_events(experiment_log_dir):
        mlflow.log_artifacts(experiment_log_dir, artifact_path="tensorboard")
        print("TensorBoard artifacts logged from:", experiment_log_dir)
    else:
        print("No TensorBoard event files found. Skipping TB artifact logging.")

# 3) Register tabular PyTorch model in Unity Catalog via MLflow UC
mlflow.set_registry_uri("databricks-uc")

try:
    uc_model_name = f"{SA.catalog_name}.{SA.schema_name}.ray_tabular_airbnb_model"
except Exception:
    uc_model_name = "main.default.ray_tabular_airbnb_model"

# Minimal MLP matching the tabular training loop (regression: output_dim=1)
class TabularMLP(nn.Module):
    def __init__(self, d_in, hidden_dim=64, d_out=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 32), nn.ReLU(),
            nn.Linear(32, d_out),
        )

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recover input_dim from train_config or feature_cols
if "train_config" in globals() and "input_dim" in train_config:
    input_dim = int(train_config["input_dim"])
elif "feature_cols" in globals():
    input_dim = len(feature_cols)
else:
    raise ValueError("Cannot infer input_dim; ensure train_config['input_dim'] or feature_cols exists.")

if result.checkpoint is None:
    raise ValueError("Ray result.checkpoint is None; cannot rebuild model for registration.")

# Rebuild and load weights from checkpoint
with result.checkpoint.as_directory() as ckpt_dir:
    weights_path = os.path.join(ckpt_dir, "model_state_dict.pt")
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"Expected weights file not found at {weights_path}")

    model_to_register = TabularMLP(d_in=input_dim, hidden_dim=int(train_config.get("hidden_dim", 64)), d_out=1).to(device).eval()
    state_dict = torch.load(weights_path, map_location="cpu")
    model_to_register.load_state_dict(state_dict)

    # Example input: 2D [N, D]
    ex = torch.randn(1, input_dim, dtype=torch.float32)
    example_input_np = ex.detach().cpu().numpy()
    signature = infer_signature(
        example_input_np,
        model_to_register(ex).detach().cpu().numpy()
    )

arch_name = "TabularMLP"
framework_tag = "pytorch-mlp-regression"

# 4) Log & register the model with MLflow UC
mlflow.end_run()
with mlflow.start_run(run_name=f"register_{run_name}") as run:
    mlflow.log_params({
        "framework": "pytorch",
        "architecture": arch_name,
        "framework_tag": framework_tag,
    })

    # If you logged regression test metrics earlier, attach them here:
    for m_name in [
        "test_rmse_log", "test_mae_log", "test_r2_log",
        "test_rmse_price", "test_mae_price", "test_r2_price",
    ]:
        if m_name in globals():
            mlflow.log_metric(m_name, float(globals()[m_name]))

    import mlflow.pytorch
    mlflow.pytorch.log_model(
        pytorch_model=model_to_register,
        name="ray_tabular_airbnb_artifact",
        input_example=example_input_np,
        signature=signature,
    )

    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/ray_tabular_airbnb_artifact"
    registered = mlflow.register_model(model_uri=model_uri, name=uc_model_name)

# 5) Optionally tag the registered model and set alias
client = MlflowClient()

# Example: tag best_metric on the model version if available
if "best_metric" in log_metrics:
    client.set_model_version_tag(
        name=uc_model_name,
        version=registered.version,
        key="best_metric",
        value=str(log_metrics["best_metric"]),
    )

# Also tag best test RMSE on price if present
if "test_rmse_price" in globals():
    client.set_model_version_tag(
        name=uc_model_name,
        version=registered.version,
        key="test_rmse_price",
        value=str(test_rmse_price),
    )

client.set_registered_model_alias(
    name=uc_model_name,
    alias="staging",
    version=registered.version,
)

print(f"Registered {uc_model_name} v{registered.version} and set alias 'staging'.")
displayHTML(
    f'<a href="/mlflow/models/{uc_model_name}/versions/{registered.version}" target="_blank">'
    "Open MLflow Model UI</a>"
)